# Document EDA and unsupervised preparation

Describe the 718,330 formal records and prepare a fixed, training-only parent-document feature cohort. Full-corpus figures use descriptive metadata. Lexical exploration uses complete derived bodies from the eligible development cohort, so paragraph boundary defects do not alter the text in this analysis.

## Bind the document handoff and exclusion ledgers

In [ ]:
import hashlib
import json
import tempfile
from collections import Counter
from pathlib import Path

import nbformat
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pyspark import StorageLevel
from pyspark.ml.feature import CountVectorizer, CountVectorizerModel, IDF, IDFModel, Normalizer
from pyspark.ml.linalg import SparseVector
from pyspark.sql import Window
from pyspark.sql import functions as F

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/06_chunk_evaluation.ipynb").is_file()
)
NOTEBOOK = ROOT / "notebooks/13_document_eda_and_unsupervised_preparation.ipynb"
n06 = nbformat.read(ROOT / "notebooks/06_chunk_evaluation.ipynb", as_version=4)
n06_cells = {cell.id: cell.source for cell in n06.cells}
handoff = {"__name__": "__main__"}
for cell_id in ("06-configure", "06-bind"):
    exec(compile(n06_cells[cell_id], f"06_chunk_evaluation.ipynb:{cell_id}", "exec"), handoff)
spark = handoff["spark"]
parents = handoff["parents"]
sha256_file = handoff["sha256_file"]
reference_count = handoff["expected"]
qa_root = ROOT / ".cache/chunk-policy-review/program-chunk-qa-v1"
qa_manifest_path = qa_root / "MANIFEST.json"
qa_manifest = json.loads(qa_manifest_path.read_text(encoding="utf-8"))
qa_audit = qa_manifest["reference_candidate_audit"]


def canonical_sha256(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(data).hexdigest()


def load_hold_table(name):
    parts = qa_manifest["tables"][name]
    folder = qa_root / name
    if {p.relative_to(qa_root).as_posix() for p in folder.glob("*.parquet")} != {
        row["path"] for row in parts
    }:
        raise ValueError(f"Unexpected input parts for {name}.")
    for row in parts:
        path = qa_root / row["path"]
        if (
            sha256_file(path) != row["sha256"]
            or pq.ParquetFile(path).metadata.num_rows != row["rows"]
        ):
            raise ValueError(f"The QA input failed verification for {name}.")
    return spark.read.parquet(str(folder))


parent_holds = load_hold_table("parent_holds")
exact_holds = load_hold_table("exact_holds")
qa_selection = load_hold_table("qa_sample")

output_base = ROOT / ".cache/corpus-eda"
output_base.mkdir(mode=0o700, exist_ok=True)
output = output_base / "document-eda-v1"
policy = {
    "cohort_seed": "document-eda-train-v1-20261122",
    "cohort_parents": 20_000,
    "minimum_document_frequency": 15,
    "maximum_vocabulary": 24_000,
    "max_document_frequency_fraction": 0.9,
    "fit_split": "train",
    "feature_unit": "document",
    "normalization": "l2",
    "feature_terms": "word_unigrams_and_bigrams_within_text_segments",
    "similarity_exclusions": [
        "whole_parent_input_hold", "exact_held_chunk_parent", "known_document_overlap",
    ],
    "within_split_duplicate_policy": "lowest_release_row_per_clean_body_sha256",
}
source_digest = canonical_sha256([
    (cell.id, cell.source) for cell in nbformat.read(NOTEBOOK, as_version=4).cells
])
identity = {
    "notebook_sources_sha256": source_digest,
    "document_handoff_manifest_sha256": sha256_file(handoff["handoff_manifest_path"]),
    "chunk_qa_manifest_sha256": sha256_file(qa_manifest_path),
    "feature_policy_sha256": canonical_sha256(policy),
}


In [ ]:
hold_pointer_path = ROOT / ".cache/corpus-freeze/SOURCE_USE_BODY_HOLDS.json"
if not hold_pointer_path.is_file():
    raise FileNotFoundError("Run Notebook 17 to verify the source-use body hold inventory.")
hold_pointer = json.loads(hold_pointer_path.read_text(encoding="utf-8"))
source_use_manifest = ROOT / hold_pointer["freeze_manifest"]
if (
    hold_pointer["version"] != "source-use-native-body-hold-v1"
    or sha256_file(source_use_manifest) != hold_pointer["freeze_manifest_sha256"]
):
    raise ValueError("The source-use hold receipt did not verify.")
hold_manifest = json.loads(source_use_manifest.read_text(encoding="utf-8"))
handoff_path = handoff["handoff_manifest_path"].relative_to(ROOT).as_posix()
if (
    hold_manifest["identity"]["version"] != "formal-corpus-release-20261117-v1"
    or hold_manifest["identity"]["source_hashes"][handoff_path]
    != identity["document_handoff_manifest_sha256"]
    or hold_manifest["source_use_screened_canonical_train_documents"] != 641_180
):
    raise ValueError("The body hold inventory does not match this document handoff.")
hold_root = source_use_manifest.parent
hold_files = hold_manifest["table_parts_sha256"]["source_use_body_holds"]
if not hold_files or any(sha256_file(hold_root / name) != sha for name, sha in hold_files.items()):
    raise ValueError("A source-use body hold file failed verification.")
source_use_body_holds = spark.read.parquet(*[str(hold_root / name) for name in hold_files])
if source_use_body_holds.select("release_row").distinct().count() != 732:
    raise ValueError("The source-use body hold roster did not reconcile.")
output = output_base / "document-eda-v2-source-use"
identity["source_use_body_hold_manifest_sha256"] = hold_pointer["freeze_manifest_sha256"]
policy["source_use_rule"] = "exclude_exact_native_bodies_pending_source_use_provenance"
identity["feature_policy_sha256"] = canonical_sha256(policy)

## Close the semantic sample receipt roster

The semantic sample can quantify the issues recorded by reviewers. Material verdicts still require adjudication before the chunk branch can be adopted. Inspect the paragraph-only estimate separately from the single-chunk control. Weighted rates apply to the frozen program-only QA frames. The receipts are reviewer judgements, and this audit does not measure inter-reviewer agreement.

In [ ]:
# Reuse the completed chunk QA receipt validator without stopping this notebook's session.
n11 = nbformat.read(ROOT / "notebooks/11_program_chunk_qa.ipynb", as_version=4)
validation_cell = next(cell.source for cell in n11.cells if cell.id == "11-qa_status")
body, terminator, trailer = validation_cell.rpartition("spark.stop()")
if not terminator or trailer.strip():
    raise ValueError("The upstream semantic QA validator changed.")
qa_namespace = {
    "output": qa_root,
    "published": qa_manifest,
    "targets": qa_audit["fresh_sample_parents"],
    "canonical_sha256": canonical_sha256,
    "json": json,
    "Counter": Counter,
    "selection": qa_selection,
    "strata": ["boundary_kind", "ml_split", "format", "length_band"],
    "spark": spark,
    "F": F,
    "display": display,
}
exec(compile(body, "11_program_chunk_qa.ipynb:11-qa_status", "exec"), qa_namespace)
reviewed_qa = qa_namespace["completed"]
if len(reviewed_qa) != sum(qa_audit["fresh_sample_parents"].values()):
    raise ValueError("The frozen QA sample does not have complete receipts.")
review_input_checksums = {
    batch: sha256_file(qa_root / "decisions" / f"{batch}.jsonl")
    for batch in qa_manifest["qa_batches"]
}
qa_counts = (
    pd.DataFrame([
        {
            "boundary_kind": packet["boundary_kind"],
            "material": decision["boundary_outcome"] == "material_evidence_break",
            "uncertain": decision["boundary_outcome"] == "uncertain",
            "evidence_loss": decision["model_view_evidence_loss"],
            "weight": packet["sampling_weight"],
        }
        for packet, decision in reviewed_qa
    ])
    .assign(weighted_material=lambda d: d.material * d.weight)
)
qa_report = qa_counts.groupby("boundary_kind", as_index=False).agg(
    reviewed=("material", "size"),
    material=("material", "sum"),
    uncertain=("uncertain", "sum"),
    exact_hold_evidence_loss=("evidence_loss", "sum"),
    weight_sum=("weight", "sum"),
    weighted_material=("weighted_material", "sum"),
)
qa_report["weighted_material_share"] = qa_report["weighted_material"] / qa_report["weight_sum"]
display(qa_report.drop(columns="weighted_material"))


## Describe the formal corpus

Count all records by year, host and format. Missing publication years remain unknown. Use code points for length so measurement is independent of a model tokenizer. The 10th and 90th percentiles summarize length dispersion.

In [ ]:
metadata = parents.select(
    "release_row", "release_group_id", "clean_sha256", "ml_split", "source_host", "format",
    "publication_year", "publication_precision", "clean_chars", "cross_split_review_required",
).persist(StorageLevel.MEMORY_AND_DISK)
census = metadata.agg(
    F.count("*").alias("documents"),
    F.count_distinct("release_row").alias("unique_documents"),
    F.count_distinct("source_host").alias("known_hosts"),
    F.count_if(F.col("publication_year").isNull()).alias("unknown_year"),
    F.percentile_approx("clean_chars", [0.1, 0.5, 0.9, 0.99], 10000).alias("length_quantiles"),
).first().asDict()
if census["documents"] != reference_count or census["unique_documents"] != reference_count:
    raise ValueError("The descriptive census does not match the formal document roster.")
years = metadata.groupBy("publication_year", "format").count().toPandas()
lengths = metadata.groupBy("format").agg(
    F.count("*").alias("documents"),
    F.percentile_approx("clean_chars", 0.1, 10000).alias("p10"),
    F.percentile_approx("clean_chars", 0.5, 10000).alias("median"),
    F.percentile_approx("clean_chars", 0.9, 10000).alias("p90"),
).toPandas()
hosts = metadata.groupBy("source_host").count().orderBy(F.desc("count"), "source_host").toPandas()
census["top_five_host_fraction"] = float(hosts.dropna(subset="source_host").head(5)["count"].sum()) / (
    census["documents"]
)
display(pd.DataFrame(census.items(), columns=["Measure", "Value"]))
display(lengths.sort_values("documents", ascending=False))


## Build the training-only feature cohort

Carry forward whole-parent holds and known document overlap flags. Conservatively exclude entire documents containing a currently held exact chunk from this feature cohort because the whole-body view would restore the held text. Retain one exact body per training split using the lowest release row. These are view-level decisions with no changes to formal membership.

Draw 20,000 canonical training documents with a fixed hash order and proportional quotas within each format and period, including every populated stratum. Preserve the stratum populations and inverse inclusion weights for lexical summaries.

In [ ]:
parent_hold_ids = parent_holds.select("release_row").withColumn("whole_parent_hold", F.lit(True))
exact_hold_ids = exact_holds.select("release_row").distinct().withColumn(
    "exact_held_chunk_parent", F.lit(True)
)
source_use_keys = (
    source_use_body_holds.select("release_row")
    if "source_use_body_holds" in globals()
    else metadata.select("release_row").limit(0)
)
source_hold_ids = source_use_keys.withColumn("source_use_body_hold", F.lit(True))
classified = (
    metadata.join(F.broadcast(parent_hold_ids), "release_row", "left")
    .join(F.broadcast(exact_hold_ids), "release_row", "left")
    .join(F.broadcast(source_hold_ids), "release_row", "left")
    .fillna(False, subset=["whole_parent_hold", "exact_held_chunk_parent", "source_use_body_hold"])
    .withColumn(
        "excluded_from_document_features",
        F.col("whole_parent_hold")
        | F.col("exact_held_chunk_parent")
        | F.col("source_use_body_hold")
        | F.coalesce("cross_split_review_required", F.lit(True)),
    )
)
eligibility = classified.groupBy("ml_split").agg(
    F.count("*").alias("formal_documents"),
    F.count_if("whole_parent_hold").alias("whole_parent_holds"),
    F.count_if("exact_held_chunk_parent").alias("exact_held_chunk_parents"),
    F.count_if("cross_split_review_required").alias("known_overlap_records"),
    F.count_if("source_use_body_hold").alias("source_use_body_holds"),
    F.count_if("excluded_from_document_features").alias("excluded_unique_parents"),
).orderBy("ml_split").toPandas()
eligible = classified.where(~F.col("excluded_from_document_features"))
if eligible.groupBy("clean_sha256").agg(
    F.min("ml_split").alias("first_split"), F.max("ml_split").alias("last_split")
).where(F.col("first_split") != F.col("last_split")).limit(1).count():
    raise ValueError("An exact document hash crosses splits in the eligible view.")
canonical_order = Window.partitionBy("clean_sha256", "ml_split").orderBy("release_row")
train = (
    eligible.where(F.col("ml_split") == policy["fit_split"])
    .withColumn("copy_index", F.row_number().over(canonical_order))
    .where(F.col("copy_index") == 1)
    .withColumn("format", F.coalesce("format", F.lit("Unspecified")))
    .withColumn(
        "period",
        F.when(F.col("publication_year").isNull(), "Unknown")
        .when(F.col("publication_year") <= 2010, "2003–2010")
        .when(F.col("publication_year") <= 2017, "2011–2017")
        .otherwise("2018–2025"),
    )
    .select("release_row", "clean_sha256", "format", "period", "publication_year", "clean_chars")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
strata = ["format", "period"]
stratum_sizes = train.groupBy(*strata).count().orderBy(*strata).collect()
population = sum(row["count"] for row in stratum_sizes)
if "source_use_body_holds" in globals() and population != 641_180:
    raise ValueError("The source-use-screened canonical training frame changed.")
target_size = policy["cohort_parents"]
if not len(stratum_sizes) <= target_size <= population:
    raise ValueError("The development cohort cannot represent all populated strata.")
remaining_target = target_size - len(stratum_sizes)
remaining_population = population - len(stratum_sizes)
apportionment = [
    divmod((row["count"] - 1) * remaining_target, remaining_population)
    for row in stratum_sizes
]
quotas = [1 + q for q, _ in apportionment]
priority = sorted(range(len(quotas)), key=lambda i: (-apportionment[i][1], i))
for i in priority[:target_size - sum(quotas)]:
    quotas[i] += 1
allocations = spark.createDataFrame(
    [
        (row.format, row.period, row["count"], quota)
        for row, quota in zip(stratum_sizes, quotas, strict=True)
    ],
    [*strata, "stratum_population", "sample_quota"],
)
selection = (
    train.join(F.broadcast(allocations), strata)
    .withColumn(
        "draw_key",
        F.sha2(F.concat_ws("|", F.lit(policy["cohort_seed"]), "clean_sha256"), 256),
    )
    .withColumn(
        "sample_rank", F.row_number().over(
            Window.partitionBy(*strata).orderBy("draw_key", "release_row")
        )
    )
    .where(F.col("sample_rank") <= F.col("sample_quota"))
    .withColumn("sampling_weight", F.col("stratum_population") / F.col("sample_quota"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
actual = selection.groupBy(*strata).count()
if actual.count() != len(stratum_sizes) or allocations.join(actual, strata, "left").where(
    F.col("count").isNull() | (F.col("count") != F.col("sample_quota"))
).limit(1).count():
    raise ValueError("The training-only cohort did not match its stratum quotas.")
eligibility["eligible_before_exact_document_canonicalization"] = (
    eligibility.formal_documents - eligibility.excluded_unique_parents
)
display(eligibility)
print(f"Eligible canonical training documents: {population:,}")
display(selection.groupBy("format").count().orderBy(F.desc("count")).toPandas())


## Explore document-level lexical evidence

Extract lowercase Unicode words, including contractions. Form bigrams within text segments separated by line or sentence-like boundaries. Retain negation, intensity and attribution in the feature text. Lexical cue counts represent presence per parent document. Report the weighted association of predeclared pairs only when at least 30 sampled documents contain both cues. The population for these estimates is the eligible canonical train view. Source mix can change across periods. These comparisons describe observed wording and do not measure fear prevalence, copying between speakers or cultural transmission.

In [ ]:
token_pattern = r"[\p{L}\p{N}]+(?:['’][\p{L}]+)*"
bodies = parents.where(F.col("ml_split") == policy["fit_split"]).select(
    "release_row", "clean_sha256", "clean_text"
)
segments = F.transform(
    F.split(F.col("clean_text"), r"(?:[.!?。！？]+\s+|\r?\n)"),
    lambda line: F.regexp_extract_all(F.lower(line), F.lit(token_pattern), F.lit(0)),
)
tokenized = (
    bodies.join(F.broadcast(selection), ["release_row", "clean_sha256"])
    .withColumn("segments", segments)
    .withColumn("words", F.flatten("segments"))
    .withColumn(
        "bigrams",
        F.flatten(F.transform(
            "segments",
            lambda part: F.zip_with(
                F.slice(part, 1, F.greatest(F.size(part) - 1, F.lit(0))),
                F.slice(part, 2, F.greatest(F.size(part) - 1, F.lit(0))),
                lambda left, right: F.concat_ws(" ", left, right),
            ),
        )),
    )
    .withColumn("terms", F.concat("words", "bigrams"))
    .drop("segments", "clean_text")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
observed = tokenized.agg(
    F.count("*").alias("documents"),
    F.count_distinct("release_row").alias("unique_documents"),
    F.count_if(F.size("words") == 0).alias("empty_lexical_documents"),
).first().asDict()
if observed["documents"] != target_size or observed["unique_documents"] != target_size:
    raise ValueError("The sampled training bodies did not bind to their expected hashes.")

cue_terms = {
    "heat": ["heat"],
    "heatwave": ["heatwave", "heatwaves", "heat wave", "heat waves"],
    "temperature": ["temperature", "temperatures"],
    "summer": ["summer", "summers"],
    "drought": ["drought", "droughts"],
    "bushfire": ["bushfire", "bushfires", "bush fire", "bush fires"],
    "climate change": ["climate change"],
    "extreme heat": ["extreme heat"],
    "rising temperatures": ["rising temperatures", "rising temperature"],
    "collective future": ["our children", "our kids", "future generations"],
}
cue_flags = tokenized.select(
    "release_row", "format", "period", "sampling_weight",
    *[
        F.arrays_overlap("terms", F.array(*map(F.lit, variants))).alias(label)
        for label, variants in cue_terms.items()
    ],
).persist(StorageLevel.MEMORY_AND_DISK)


def cue_rates(group):
    tall = cue_flags.select(
        group, "sampling_weight",
        F.explode(F.array(*[
            F.struct(F.lit(label).alias("cue"), F.col(label).alias("present"))
            for label in cue_terms
        ])).alias("evidence"),
    ).select(group, "sampling_weight", "evidence.*")
    return tall.groupBy(group, "cue").agg(
        F.count("*").alias("sample_documents"),
        F.count_if("present").alias("documents_with_cue"),
        F.sum("sampling_weight").alias("estimated_population_documents"),
        (
            F.sum(F.when(F.col("present"), F.col("sampling_weight")).otherwise(0.0))
            / F.sum("sampling_weight")
        ).alias("weighted_document_share"),
    ).orderBy(group, "cue").toPandas()


period_cues = cue_rates("period")
format_cues = cue_rates("format")
# Predeclared document-level pairs compare association against each marginal document rate.
pairs = [
    ("heatwave", "climate change"), ("heatwave", "collective future"),
    ("heat", "summer"), ("extreme heat", "climate change"), ("temperature", "climate change"),
    ("drought", "bushfire"), ("bushfire", "climate change"),
]
exploded_pairs = cue_flags.select(
    "sampling_weight",
    F.explode(F.array(*[
        F.struct(
            F.lit(first).alias("first"), F.lit(second).alias("second"),
            F.col(first).alias("has_first"), F.col(second).alias("has_second"),
        )
        for first, second in pairs
    ])).alias("pair"),
).select("sampling_weight", "pair.*")
joint = F.col("has_first") & F.col("has_second")
pair_stats = (
    exploded_pairs.groupBy("first", "second")
    .agg(
        F.count("*").alias("sample_documents"),
        F.count_if(joint).alias("observed_joint"),
        F.sum("sampling_weight").alias("population_weight"),
        F.sum(F.when(F.col("has_first"), F.col("sampling_weight")).otherwise(0.0)).alias(
            "first_weight"
        ),
        F.sum(F.when(F.col("has_second"), F.col("sampling_weight")).otherwise(0.0)).alias(
            "second_weight"
        ),
        F.sum(F.when(joint, F.col("sampling_weight")).otherwise(0.0)).alias("joint_weight"),
    )
    .withColumn(
        "weighted_pmi_bits",
        F.when(
            F.col("observed_joint") >= 30,
            F.log2(
                (F.col("joint_weight") / F.col("first_weight"))
                * (F.col("population_weight") / F.col("second_weight"))
            ),
        ),
    )
    .orderBy(F.desc("observed_joint"))
    .toPandas()
)
display(pair_stats[["first", "second", "observed_joint", "weighted_pmi_bits"]])


## Prepare sparse TF–IDF features

Fit the development vocabulary and inverse document frequencies to the fixed training cohort. Record the vocabulary and preserve documents with zero retained features for diagnosis. A unit L2 norm puts nonzero document profiles on a comparable scale for an exploratory sparse baseline.

The next exploratory fit can compare NMF components across seeds and a small rank range using the same sparse document matrix. Read the most weighted terms together with representative documents, counterexamples and source concentrations. Compare stable topics across periods with denominators and source controls. Keep validation and test documents outside vocabulary, rank and threshold selection. Report later held-out comparisons and semantic judgements separately.

In [ ]:
# Prepare a reproducible feature space on the development cohort.
if output.exists():
    sealed = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
    if (
        sealed["identity"] != identity or sealed["chunk_qa_receipts"] != review_input_checksums
    ):
        raise ValueError("The sealed feature snapshot belongs to a different input or notebook.")
    for entry in sealed["files"]:
        if sha256_file(output / entry["path"]) != entry["sha256"]:
            raise ValueError("A sealed feature or EDA file failed its checksum.")
    cv = CountVectorizerModel.load(str(output / "count_vectorizer"))
    idf = IDFModel.load(str(output / "idf"))
    counts = cv.transform(tokenized).select("release_row", "clean_sha256", "term_counts")
else:
    cv = CountVectorizer(
        inputCol="terms",
        outputCol="term_counts",
        vocabSize=policy["maximum_vocabulary"],
        minDF=float(policy["minimum_document_frequency"]),
        maxDF=policy["max_document_frequency_fraction"],
        binary=False,
    ).fit(tokenized)
    counts = cv.transform(tokenized).select("release_row", "clean_sha256", "term_counts")
    idf = IDF(inputCol="term_counts", outputCol="raw_tfidf").fit(counts)
normalizer = Normalizer(inputCol="raw_tfidf", outputCol="tfidf", p=2.0)
nonzero = F.udf(lambda vector: int(vector.numNonzeros()), "int")
features = (
    normalizer.transform(idf.transform(counts))
    .select("release_row", "clean_sha256", "tfidf")
    .withColumn("nonzero_features", nonzero("tfidf"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
feature_checks = features.agg(
    F.count("*").alias("rows"),
    F.count_distinct("release_row").alias("unique_documents"),
    F.count_if(F.col("nonzero_features") == 0).alias("zero_feature_documents"),
    F.sum("nonzero_features").alias("nonzero_values"),
    F.percentile_approx("nonzero_features", 0.5, 10000).alias("median_nonzero_features"),
).first().asDict()
vocab = cv.vocabulary
document_frequencies = idf.docFreq
if (
    not vocab
    or feature_checks["rows"] != target_size
    or feature_checks["unique_documents"] != target_size
    or len(document_frequencies) != len(vocab)
    or idf.numDocs != target_size
):
    raise ValueError("Sparse features did not reconcile to the sampled training documents.")
initial_vector = features.where(F.col("nonzero_features") > 0).select("tfidf").first()[0]
if not isinstance(initial_vector, SparseVector) or initial_vector.size != len(vocab):
    raise ValueError("The feature space did not remain sparse or match its vocabulary.")
vocabulary = pd.DataFrame({
    "index": range(len(vocab)),
    "term": vocab,
    "document_frequency": document_frequencies,
    "inverse_document_frequency": list(idf.idf),
})
feature_checks["vocabulary_size"] = len(vocab)
feature_checks["canonical_eligible_training_frame"] = population
display(pd.DataFrame(feature_checks.items(), columns=["Measure", "Value"]))
display(vocabulary.nlargest(12, "document_frequency")[["term", "document_frequency"]])


## Visualize coverage, length and lexical cues

Use census denominators for source, date and length panels, and training-stratum inclusion weights for the lexical panel.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator, PercentFormatter, StrMethodFormatter

colors = ["#225a83", "#008578", "#b36b1e", "#a04764", "#557248", "#526585", "#75665a"]
figure, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

year_counts = (
    years.dropna(subset="publication_year").groupby("publication_year")["count"].sum().sort_index()
)
axes[0, 0].bar(year_counts.index.astype(int), year_counts.values, color=colors[0], width=0.84)
axes[0, 0].set(title="Document coverage by known publication year", ylabel="Documents")
axes[0, 0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[0, 0].text(
    0.02, 0.95, f"Unknown years: {census['unknown_year']:,}",
    transform=axes[0, 0].transAxes, va="top", color="#444444",
)

top_hosts = hosts.dropna(subset="source_host").head(8).sort_values("count")
axes[0, 1].barh(top_hosts["source_host"], top_hosts["count"], color=colors[1])
axes[0, 1].set(title="Largest source hosts", xlabel="Documents")
axes[0, 1].xaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
axes[0, 1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

main_formats = lengths.nlargest(7, "documents").sort_values("median").reset_index(drop=True)
axes[1, 0].hlines(
    main_formats.index, main_formats.p10, main_formats.p90, color=colors[0], linewidth=4, alpha=0.48
)
axes[1, 0].scatter(main_formats["median"], main_formats.index, color=colors[0], s=45, zorder=2)
axes[1, 0].set_yticks(
    main_formats.index, main_formats.apply(lambda r: f"{r['format']} ({r['documents']:,})", axis=1)
)
axes[1, 0].set_xscale("log")
axes[1, 0].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
axes[1, 0].set(
    title="Text length by format", xlabel="Unicode code points (10th, median and 90th percentile)"
)

shown_periods = ["2003–2010", "2011–2017", "2018–2025"]
shown_cues = ["heat", "heatwave", "temperature", "bushfire", "drought"]
matrix = (
    period_cues[period_cues.cue.isin(shown_cues)]
    .pivot(index="period", columns="cue", values="weighted_document_share")
    .reindex(index=shown_periods, columns=shown_cues)
)
matrix.plot.bar(ax=axes[1, 1], color=colors[:5], width=0.75, rot=0)
axes[1, 1].set(
    title="Selected lexical cues in the training cohort", xlabel="Publication period",
    ylabel="Weighted share of documents",
)
axes[1, 1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1, 1].legend(frameon=False, fontsize=8, ncols=2)
for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)

display(figure)
plt.close(figure)


## Save the local exploratory snapshot

Store aggregate tables, figures, the development cohort, fitted vocabulary and sparse TF–IDF features. The local sample ledger and model feature files carry row-level provenance. Corpus text remains in the bound Notebook 05 handoff.

In [ ]:
aggregate_tables = {
    "year_by_format.csv": years.sort_values(["publication_year", "format"], na_position="last"),
    "length_by_format.csv": lengths.sort_values("format"),
    "host_counts.csv": hosts,
    "document_feature_eligibility.csv": eligibility,
    "train_cohort_strata.csv": allocations.orderBy(*strata).toPandas(),
    "train_cues_by_period.csv": period_cues,
    "train_cues_by_format.csv": format_cues,
    "train_document_cue_pairs.csv": pair_stats,
    "chunk_qa_by_boundary_kind.csv": qa_report,
    "vocabulary.csv": vocabulary,
}
if not output.exists():
    with tempfile.TemporaryDirectory(prefix=".document-eda-", dir=output_base) as temporary:
        stage = Path(temporary) / "snapshot"
        stage.mkdir(mode=0o700)
        for name, table in aggregate_tables.items():
            table.to_csv(stage / name, index=False)
        figure.savefig(stage / "corpus_eda.png", dpi=220, bbox_inches="tight", facecolor="white")
        figure.savefig(stage / "corpus_eda.pdf", bbox_inches="tight", facecolor="white")
        (
            selection.select(
                "release_row", "clean_sha256", "period", "format", "publication_year",
                "stratum_population", "sample_quota", "sampling_weight", "draw_key",
            ).coalesce(4).write.mode("errorifexists").parquet(str(stage / "train_cohort"))
        )
        features.coalesce(4).write.mode("errorifexists").parquet(str(stage / "train_sparse_tfidf"))
        cv.write().save(str(stage / "count_vectorizer"))
        idf.write().save(str(stage / "idf"))
        (
            stage / "MANIFEST.json"
        ).write_text(
            json.dumps(
                {
                    "identity": identity,
                    "policy": policy,
                    "formal_census": census,
                    "training_cohort_checks": observed,
                    "feature_checks": feature_checks,
                    "chunk_qa_receipts": review_input_checksums,
                    "chunk_qa_status": "sample_complete_material_breaks_require_adjudication",
                    "topic_model_status": "feature_preparation_complete_model_not_fit",
                    "files": [
                        {"path": path.relative_to(stage).as_posix(), "sha256": sha256_file(path)}
                        for path in sorted(stage.rglob("*"))
                        if path.is_file() and not path.name.startswith(".")
                    ],
                },
                ensure_ascii=False, indent=2, sort_keys=True,
            ) + "\n",
            encoding="utf-8",
        )
        if output.exists():
            raise FileExistsError("Another writer created the exploratory snapshot.")
        stage.rename(output)
saved = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
if (
    saved["identity"] != identity
    or saved["feature_checks"] != feature_checks
    or saved["chunk_qa_receipts"] != review_input_checksums
):
    raise ValueError("The existing exploratory snapshot does not match the verified analysis.")
for entry in saved["files"]:
    if sha256_file(output / entry["path"]) != entry["sha256"]:
        raise ValueError("A written EDA output failed its checksum.")
for table in ("train_sparse_tfidf", "train_cohort"):
    if sum(pq.ParquetFile(p).metadata.num_rows for p in (output / table).glob("*.parquet")) != (
        target_size
    ):
        raise ValueError(f"The saved {table} row count did not match the frozen cohort.")
print(f"Verified the census and local sparse feature snapshot at {output.relative_to(ROOT)}.")
spark.stop()
